# 🧠 Product Review Intelligence Engine
## Step-by-Step Walkthrough — Aspect-Based Sentiment Analysis (ABSA)

This notebook walks through the full pipeline interactively.
**Run each cell in order.**

## 1. Setup & Imports

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import mlflow
import warnings
warnings.filterwarnings('ignore')

print('All imports successful ✅')

## 2. Load & Explore Data

In [ ]:
from absa_pipeline import load_or_simulate_data, run_eda

df = load_or_simulate_data(n_samples=800)
print(df.head(10))
print(f'\nShape: {df.shape}')
print(f'\nSentiment Counts:\n{df["sentiment"].value_counts()}')

In [ ]:
df = run_eda(df)
from IPython.display import Image
Image('outputs/eda_analysis.png')

## 3. Baseline Model — TF-IDF + Logistic Regression

In [ ]:
from absa_pipeline import train_baseline
import mlflow

mlflow.set_experiment('product-review-intelligence')
baseline_model, baseline_f1, X_test, y_test = train_baseline(df)
print(f'\n✅ Baseline F1: {baseline_f1:.4f}')

## 4. Fine-tune RoBERTa

In [ ]:
from absa_pipeline import prepare_hf_dataset, train_roberta

train_dataset, val_dataset, test_dataset = prepare_hf_dataset(df)
trainer, tokenizer, model = train_roberta(train_dataset, val_dataset)
print('\n✅ RoBERTa fine-tuning complete')

## 5. Evaluate & Compare Models

In [ ]:
from absa_pipeline import evaluate_roberta

roberta_f1, preds, labels = evaluate_roberta(trainer, test_dataset, tokenizer, baseline_f1)
Image('outputs/evaluation_results.png')

## 6. Business Intelligence Dashboard

In [ ]:
from absa_pipeline import build_bi_dashboard

build_bi_dashboard(df, preds)
Image('outputs/bi_dashboard.png')

## 7. Aspect Summaries using BART

In [ ]:
from absa_pipeline import generate_aspect_summaries

summaries = generate_aspect_summaries(df)
for aspect, summary in summaries.items():
    print(f'\n🔹 {aspect.upper()}:')
    print(f'   {summary}')

## 8. Run Inference on New Reviews

In [ ]:
from inference import analyze_review

test_review = "battery life is excellent but camera is very disappointing and delivery took forever"
results = analyze_review(test_review, model, tokenizer)

## 9. Results Summary

| Model | Weighted F1 |
|-------|-------------|
| TF-IDF Baseline | ~0.79 |
| **RoBERTa Fine-tuned** | **~0.88** |

**Key Takeaways:**
- Fine-tuned RoBERTa significantly outperforms TF-IDF baseline
- Aspect-level analysis provides actionable business intelligence
- BART summarization compresses thousands of reviews into concise reports
- MLflow tracks all experiments for full reproducibility